# Playground

The exercises check one specific answer. This notebook is for the other half
of learning: changing a number and seeing what happens.

Nothing here is graded and nothing here is checked. Break it freely; `git
checkout notebooks/playground.ipynb` puts it back.

**Kernel setup.** In VS Code the notebook kernel is a *separate* setting from
the Python interpreter. Click the kernel picker in the top right and choose the
one inside `.venv`. Getting a `ModuleNotFoundError` on `qiskit` here while
`uv run qx doctor` passes means exactly this: right interpreter, wrong kernel.

## What is installed

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("qiskit             ", qiskit.__version__)
print("qiskit-aer         ", qiskit_aer.__version__)
print("qiskit-ibm-runtime ", qiskit_ibm_runtime.__version__)

## A circuit, drawn

`draw()` with no argument prints text and always works. `draw("mpl")` needs
the visualization extra, which this project already installs.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

print(qc.draw())
qc.draw("mpl")

## The state behind it

This is the view you lose the moment you touch real hardware, so use it while
you can. Try adding a `z` or an `s` and watch which amplitude changes sign.

In [ ]:
from qiskit.quantum_info import Statevector

state = Statevector(qc)
print(state)
print()
print("probabilities:", state.probabilities_dict())

## Sampling it

Change `shots` and re-run a few times. The counts move around, and the size of
that movement is what the runner's 4-sigma tolerance is built around:
`SE = sqrt(p(1-p)/N)`.

In [ ]:
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

measured = qc.copy()
measured.measure_all()

result = StatevectorSampler().run([measured], shots=1024).result()
counts = result[0].data.meas.get_counts()
print(counts)

plot_histogram(counts)

## Interference, the exercise 09 idea, free to poke at

`H H` is the identity and `H Z H` is a NOT. Replace `z` with `s`, `t`, or
`p(angle)` and watch the certainty dissolve into a genuine 50/50 and back.

In [ ]:
import numpy as np
from qiskit.quantum_info import Operator

for middle in ["none", "z", "s", "t"]:
    c = QuantumCircuit(1)
    c.h(0)
    if middle != "none":
        getattr(c, middle)(0)
    c.h(0)
    probs = Statevector(c).probabilities_dict()
    print(f"{middle:>5}  P(0)={probs.get('0', 0):.3f}  P(1)={probs.get('1', 0):.3f}")

## Gates as matrices

In [ ]:
np.set_printoptions(precision=3, suppress=True)
print(Operator(qc).data.real)

## Which backend would exercise 13 reach?

This asks the same question the hardware exercise asks, and submits nothing.
Set `QX_OFFLINE=1` in the environment to force the offline answer.

In [ ]:
from quantum_exercises.backends import get_backend

selection = get_backend(min_num_qubits=2)
print(selection.describe())
print("reason:", selection.reason)
print("native gates:", sorted(selection.backend.target.operation_names))

## Transpiling to that backend

The step that is mandatory before hardware. Watch the Hadamard disappear into
whatever rotations the machine actually implements.

In [ ]:
from quantum_exercises.backends import to_isa

isa = to_isa(measured, selection.backend)
print("before:", dict(measured.count_ops()))
print("after :", dict(isa.count_ops()))